In [ ]:
import numpy
import pandas
import scipy

import matplotlib
import matplotlib.pyplot as plt
plt.style.use('../mystyle.mplstyle')

import effana

# inputs
# west data | east data | start run | end run
TRIG_FILES = {
  'Run1': dict(west='/Users/triozzi/Analysis/icarus-trigger/data/eff/Run1_Run2_StoppingMuon_TrigEmu_WestCryo.txt', east='/Users/triozzi/Analysis/icarus-trigger/data/eff/Run1_Run2_StoppingMuon_TrigEmu_EastCryo.txt', start=8650, end=8650),
  'Run2': dict(west='/Users/triozzi/Analysis/icarus-trigger/data/eff/Run1_Run2_StoppingMuon_TrigEmu_WestCryo.txt', east='/Users/triozzi/Analysis/icarus-trigger/data/eff/Run1_Run2_StoppingMuon_TrigEmu_EastCryo.txt', start=9465, end=9576),
  'Run3': dict(west='/Users/triozzi/Analysis/icarus-trigger/data/eff/Run2_StoppingMuon_TrigEmu_WestCryo.txt', east='/Users/triozzi/Analysis/icarus-trigger/data/eff/Run2_StoppingMuon_TrigEmu_EastCryo.txt', start=11000, end=12000),
  'Run4': dict(west='/Users/triozzi/Analysis/icarus-trigger/data/eff/Run4_StoppingMuon_TrigEmu_WestCryo.txt', east='/Users/triozzi/Analysis/icarus-trigger/data/eff/Run4_StoppingMuon_TrigEmu_EastCryo.txt', start=12000, end=14000),
  'Run5': dict(west='/Users/triozzi/Analysis/icarus-trigger/data/eff/Run5_StoppingMuon_TrigEmu_WestCryo.txt', east='/Users/triozzi/Analysis/icarus-trigger/data/eff/Run5_StoppingMuon_TrigEmu_EastCryo.txt', start=13000, end=14000),
}

# grab the KE vs. range PDG table...
df = pandas.read_csv(
  "/Users/triozzi/Analysis/icarus-trigger/data/input/KEVsRange_Muon_PDGTable.txt", 
  sep='\s+', 
  index_col=False, 
  on_bad_lines='skip',
  skiprows=1,
  names=["T", "p", "ionization", "brems", "pair", "photonuc", "radloss", "dEdx", "range", "delta", "beta", "dEdx_r"]
)
f2 = scipy.interpolate.interp1d(df["range"]/1.396, df["T"], kind='cubic')

# dump all plots here, and gitignore it
PRINT_PATH = 'plots/'

#### Run1 and Run2

In [ ]:
# load data
df = effana.process.ImportData(TRIG_FILES['Run1'], cryo='both', run='Run1')

# clean up data pretty aggressively, but in a very clean way
df = effana.process.ApplyOfflineSelection(df)

TRIGGERS = [df.S5, df.S10] # triggers you want to analyze
VARIABLE = f2(df.length)   # variable here is the KE from range
INTERVALS = [[70, 200], [200, 300], [300, 400], [400, 500], [500, 600], [600, 1000]] # intervals to bin the variable of interest

x, x_err, run1, run1_err, c_run1 = effana.process.ComputeEfficiencyEnergy(
  TRIGGERS,
  INTERVALS,
  VARIABLE
)

# add systematic uncertainty
run1_systs, run1_err_systs = effana.systs.AddSystematicUncertainty(run1, run1_err)

In [ ]:
df = effana.process.ImportData(TRIG_FILES['Run2'], cryo='both', run='Run2')
df = effana.process.ApplyOfflineSelection(df)

TRIGGERS = [df.S5, df.S10] # triggers you want to analyze
VARIABLE = f2(df.length)   # variable here is the KE from range
INTERVALS = [[70, 200], [200, 300], [300, 400], [400, 500], [500, 600], [600, 1000]] # intervals to bin the variable of interest

x, x_err, run2, run2_err, c_run2 = effana.process.ComputeEfficiencyEnergy(
  TRIGGERS,
  INTERVALS,
  VARIABLE
)

# add systematic uncertainty
run2_systs, run2_err_systs = effana.systs.AddSystematicUncertainty(run2, run2_err)

In [ ]:
df = effana.process.ImportData(TRIG_FILES['Run2'], cryo='west', run='Run2')
df = effana.process.ApplyOfflineSelection(df)

TRIGGERS = [df.S5, df.S10] # triggers you want to analyze
VARIABLE = f2(df.length)   # variable here is the KE from range
INTERVALS = [[70, 200], [200, 300], [300, 400], [400, 500], [500, 600], [600, 1000]] # intervals to bin the variable of interest

x, x_err, run2, run2_err, c_run2 = effana.process.ComputeEfficiencyEnergy(
  TRIGGERS,
  INTERVALS,
  VARIABLE
)

# add systematic uncertainty
run2_west_systs, run2_west_err_systs = effana.systs.AddSystematicUncertainty(run2, run2_err)

In [ ]:
df = effana.process.ImportData(TRIG_FILES['Run2'], cryo='east', run='Run2')
df = effana.process.ApplyOfflineSelection(df)

TRIGGERS = [df.S5, df.S10] # triggers you want to analyze
VARIABLE = f2(df.length)   # variable here is the KE from range
INTERVALS = [[70, 200], [200, 300], [300, 400], [400, 500], [500, 600], [600, 1000]] # intervals to bin the variable of interest

x, x_err, run2, run2_err, c_run2 = effana.process.ComputeEfficiencyEnergy(
  TRIGGERS,
  INTERVALS,
  VARIABLE
)

# add systematic uncertainty
run2_east_systs, run2_east_err_systs = effana.systs.AddSystematicUncertainty(run2, run2_err)

In [ ]:
# set up the plot
fig, ax = plt.subplots(figsize=(4.25, 3.25), ncols=1, layout='constrained')

ax.errorbar(
  x=x, xerr=x_err, y=run1_systs, yerr=run1_err_systs, 
  ls='', lw=1.5, ms=5, capsize=2, 
  ecolor='C0', mec='black', color='C0', marker='o', label='Run1', zorder=1
)

ax.errorbar(
  x=x, xerr=x_err, y=run2_systs, yerr=run2_err_systs, 
  ls='', lw=1.5, ms=5, capsize=2, 
  ecolor='C1', mec='black', color='C1', marker='D', label='Run2', zorder=2
)

# gfx
ax.axhline(1, lw=0.75, c='black', zorder=-3)
ax.set(
  xlabel = 'reconstructed energy [MeV]',
  ylabel = 'mj-5 trigger efficiency',
  title = 'ICARUS data',
  ylim = (0, 1.1),
  xlim = (70, 1000)
)
ax.set_yticks([0, 0.5, 1])
ax.legend(fontsize=10)

# references
ax.axvline(490, lw=0.75, c='gray', ls='--')
ax.text(490, 0.05, 'NuMI $\\nu_{\\mu}$CC $1^\\text{st}$ $E_{\\nu}$ peak',
        color='gray', rotation=90, ha='right', va='bottom',
        transform=ax.get_xaxis_transform(), fontsize=10)

ax.axvline(970, lw=0.75, c='black')
ax.text(970, 0.05, 'BNB $\\nu_{\\mu}$CC $E_{\\nu}$ peak',
        color='black', rotation=90, ha='right', va='bottom',
        transform=ax.get_xaxis_transform(), fontsize=10)

PLOT_NAME = 'trigger_efficiency_Run1Run2'
fig.savefig(f'{PRINT_PATH}{PLOT_NAME}.pdf', dpi=300)
fig.savefig(f'{PRINT_PATH}{PLOT_NAME}.png', dpi=300)

#### from Run3 onward 

Add the adder contribution!

In [ ]:
df = effana.process.ImportData(TRIG_FILES['Run3'], cryo='both', run='Run3')
df = effana.process.ApplyOfflineSelection(df)

TRIGGERS = [df.S4, df.S5, df.S10] # triggers you want to analyze
VARIABLE = f2(df.length)   # variable here is the KE from range
INTERVALS = [[70, 200], [200, 300], [300, 400], [400, 500], [500, 600], [600, 1000]] # intervals to bin the variable of interest

x, x_err, run3, run3_err, c_run3 = effana.process.ComputeEfficiencyEnergy(
  TRIGGERS,
  INTERVALS,
  VARIABLE
)

# add systematic uncertainty
run3_systs, run3_err_systs = effana.systs.AddSystematicUncertainty(run3, run3_err)

In [ ]:
df = effana.process.ImportData(TRIG_FILES['Run4'], cryo='both', run='Run4')
df = effana.process.ApplyOfflineSelection(df)

TRIGGERS = [df.S4, df.S5, df.S10] # triggers you want to analyze
VARIABLE = f2(df.length)   # variable here is the KE from range
INTERVALS = [[70, 200], [200, 300], [300, 400], [400, 500], [500, 600], [600, 1000]] # intervals to bin the variable of interest

x, x_err, run4, run4_err, c_run4 = effana.process.ComputeEfficiencyEnergy(
  TRIGGERS,
  INTERVALS,
  VARIABLE
)

# add systematic uncertainty
run4_systs, run4_err_systs = effana.systs.AddSystematicUncertainty(run4, run4_err, TRIGGER_IDX=0, APPLY_MISCONFIG=False)

In [ ]:
df = effana.process.ImportData(TRIG_FILES['Run5'], cryo='both', run='Run5')
df = effana.process.ApplyOfflineSelection(df)

TRIGGERS = [df.S4, df.S5, df.S10] # triggers you want to analyze
VARIABLE = f2(df.length)   # variable here is the KE from range
INTERVALS = [[70, 200], [200, 300], [300, 400], [400, 500], [500, 600], [600, 1000]] # intervals to bin the variable of interest

x, x_err, run5, run5_err, c_run5 = effana.process.ComputeEfficiencyEnergy(
  TRIGGERS,
  INTERVALS,
  VARIABLE
)

# add systematic uncertainty
run5_systs, run5_err_systs = effana.systs.AddSystematicUncertainty(run5, run5_err, TRIGGER_IDX=0, APPLY_MISCONFIG=False)

In [ ]:
# set up the plot
fig, ax = plt.subplots(figsize=(4.25, 3.25), ncols=1, layout='constrained')

ax.errorbar(
  x=x, xerr=x_err, y=run1_systs, yerr=run1_err_systs, 
  ls='', lw=1.5, ms=5, capsize=2, 
  ecolor='C0', mec='black', color='C0', marker='o', label='Run1\n(mj-5)', zorder=1
)

ax.errorbar(
  x=x, xerr=x_err, y=run2_systs, yerr=run2_err_systs, 
  ls='', lw=1.5, ms=5, capsize=2, 
  ecolor='C1', mec='black', color='C1', marker='D', label='Run2\n(mj-5)', zorder=2
)

ax.errorbar(
  x=x, xerr=x_err, y=run3_systs * effana.config.ADDER_FACTORS['Run3'], yerr=run3_err_systs, 
  ls='', lw=1.5, ms=5, capsize=2, 
  ecolor='C2', mec='black', color='C2', marker='d', label='Run3\n(mj-4 + add)', zorder=3
)

ax.errorbar(
  x=x, xerr=x_err, y=run4_systs * effana.config.ADDER_FACTORS['Run4'], yerr=run4_err_systs, 
  ls='', lw=1.5, ms=5, capsize=2, 
  ecolor='C3', mec='black', color='C3', marker='s', label='Run4\n(mj-4 + add)', zorder=4
)

ax.errorbar(
  x=x, xerr=x_err, y=run5_systs * effana.config.ADDER_FACTORS['Run5'], yerr=run5_err_systs, 
  ls='', lw=1.5, ms=5, capsize=2, 
  ecolor='C4', mec='black', color='C4', marker='p', label='Run5\n(mj-4 + add)', zorder=4
)

# gfx
ax.axhline(1, lw=0.75, c='black', zorder=-3)
ax.set(
  xlabel = 'reconstructed energy [MeV]',
  ylabel = 'trigger efficiency',
  title = 'ICARUS data',
  ylim = (0, 1.1),
  xlim = (70, 1000)
)
ax.set_yticks([0, 0.5, 1])
ax.legend(fontsize=9.5, loc=(0.5, 0.08))

# references
ax.axvline(490, lw=0.75, c='gray', ls='--')
ax.text(490, 0.05, 'NuMI $\\nu_{\\mu}$CC $1^\\text{st}$ $E_{\\nu}$ peak',
        color='gray', rotation=90, ha='right', va='bottom',
        transform=ax.get_xaxis_transform(), fontsize=10)

ax.axvline(970, lw=0.75, c='black')
ax.text(970, 0.05, 'BNB $\\nu_{\\mu}$CC $E_{\\nu}$ peak',
        color='black', rotation=90, ha='right', va='bottom',
        transform=ax.get_xaxis_transform(), fontsize=10)

PLOT_NAME = 'trigger_efficiency_Run1Run2Run3Run4Run5'
fig.savefig(f'{PRINT_PATH}{PLOT_NAME}.pdf', dpi=300)
fig.savefig(f'{PRINT_PATH}{PLOT_NAME}.png', dpi=300)

In [ ]:
run5_systs * effana.config.ADDER_FACTORS['Run5']